# 03 — Self-Supervised Training: Autoencoder Change Detection

Train a convolutional autoencoder to reconstruct satellite images. At inference, high reconstruction error on different-date pairs indicates change — **no labels needed**.

**Key idea**: Same-date images reconstruct well (learned distribution). Different-date images reconstruct poorly (out-of-distribution → change).

In [ ]:
!rm -rf delta
GIT_TERMINAL_PROMPT=0 git clone https://github.com/1046prt/delta.git
!pip install torchgeo matplotlib numpy tqdm -q

In [ ]:
import os
import sys
import time
import tempfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torchgeo.datasets import OSCD100
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 0. Fix Windows paths (needed when dataset was downloaded on Windows)

In [ ]:
def fix_windows_paths(root_dir):
    """Rename files/dirs with backslashes to forward slashes for Linux compatibility."""
    count = 0
    for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
        for name in filenames:
            if '\\' in name:
                old = os.path.join(dirpath, name)
                new = os.path.join(dirpath, name.replace('\\', '/'))
                os.rename(old, new)
                count += 1
        for name in dirnames:
            if '\\' in name:
                old = os.path.join(dirpath, name)
                new = os.path.join(dirpath, name.replace('\\', '/'))
                os.rename(old, new)
                count += 1
    print(f'Fixed {count} paths')

root = os.path.join(tempfile.gettempdir(), 'oscd100')
fix_windows_paths(root)

## 1. Setup: Clone repo and import modules

In [ ]:
if os.path.exists('delta/src'):
    sys.path.insert(0, 'delta')
    print('Using cloned repo')
elif os.path.exists('../src'):
    sys.path.insert(0, '..')
    print('Using parent dir')
else:
    print('Modules not found - check clone step')

In [ ]:
from src.models.unet import Autoencoder
from src.training.metrics import compute_f1, compute_iou

print("Imports OK")

## 2. Dataset

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'oscd100')

class OSCDReconstruction(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image'].float()
        mask = sample['mask'].float()

        if image.dim() == 4:
            t1, t2 = image[0], image[1]
        else:
            t1 = t2 = image

        return {'t1': t1, 't2': t2, 'mask': mask}


train_dataset = OSCD100(root=root, split='train', bands=OSCD100.rgb_bands, download=True)
val_dataset = OSCD100(root=root, split='val', bands=OSCD100.rgb_bands, download=True)

train_ds = OSCDReconstruction(train_dataset)
val_ds = OSCDReconstruction(val_dataset)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
sample = train_ds[0]
print(f"T1 shape: {sample['t1'].shape}, T2 shape: {sample['t2'].shape}")

## 3. Autoencoder Model

In [ ]:
model = Autoencoder(
    in_channels=3,   # RGB only (OSCD100.rgb_bands = B04, B03, B02)
    features=[64, 128, 256, 512],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Autoencoder parameters: {n_params:,}")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

## 4. Training (Reconstruction Loss)

In [ ]:
MAX_EPOCHS = 60
USE_AMP = device.type == 'cuda'
scaler = GradScaler() if USE_AMP else None
CHANGE_THRESHOLD = 0.15

best_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}

for epoch in range(MAX_EPOCHS):
    t0 = time.time()

    model.train()
    train_loss = 0.0
    for batch in train_loader:
        t1 = batch['t1'].to(device)
        t2 = batch['t2'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            recon_t1 = model(t1)
            recon_t2 = model(t2)
            loss = criterion(recon_t1, t1) + criterion(recon_t2, t2)

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    all_f1 = []
    all_iou = []
    with torch.no_grad():
        for batch in val_loader:
            t1 = batch['t1'].to(device)
            t2 = batch['t2'].to(device)
            masks = batch['mask'].to(device)

            recon_t1 = model(t1)
            recon_t2 = model(t2)
            val_loss += (criterion(recon_t1, t1) + criterion(recon_t2, t2)).item()

            err_t1 = torch.mean((t1 - recon_t1)**2, dim=1, keepdim=True)
            err_t2 = torch.mean((t2 - recon_t2)**2, dim=1, keepdim=True)
            change_map = torch.abs(err_t1 - err_t2)
            change_pred = (change_map > CHANGE_THRESHOLD).float()

            all_f1.append(compute_f1(change_pred, masks))
            all_iou.append(compute_iou(change_pred, masks))

    val_loss /= len(val_loader)
    mean_f1 = np.mean(all_f1)
    mean_iou = np.mean(all_iou)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(mean_f1)
    history['val_iou'].append(mean_iou)

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1:3d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | "
        f"Change F1: {mean_f1:.4f} | IoU: {mean_iou:.4f} | "
        f"LR: {lr:.2e} | {elapsed:.1f}s"
    )

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_loss': best_loss,
            'config': {
                'model': {'type': 'autoencoder', 'in_channels': 3, 'features': [64, 128, 256, 512]},
                'data': {'bands': ['B04', 'B03', 'B02'], 'use_ndvi': False},
                'inference': {'change_threshold': CHANGE_THRESHOLD},
            },
        }, 'autoencoder_best.pt')
        print(f"  -> Saved best model (Loss: {best_loss:.6f})")
    else:
        patience_counter += 1

    if patience_counter >= 15:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best reconstruction loss: {best_loss:.6f}")

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Reconstruction Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_f1'], label='Change F1', linewidth=2, color='tab:blue')
axes[1].plot(epochs, history['val_iou'], label='Change IoU', linewidth=2, color='tab:orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Change Detection Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Self-Supervised Autoencoder Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_selfsupervised.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Visualize Change Detection

In [ ]:
model.eval()
n_show = min(4, len(val_ds))

fig, axes = plt.subplots(n_show, 5, figsize=(25, 5*n_show))

for i in range(n_show):
    sample = val_ds[i]
    t1 = sample['t1'].unsqueeze(0).to(device)
    t2 = sample['t2'].unsqueeze(0).to(device)
    mask_gt = sample['mask'].squeeze().numpy()

    with torch.no_grad():
        recon_t1 = model(t1)
        recon_t2 = model(t2)

    err_t1 = torch.mean((t1 - recon_t1)**2, dim=1, keepdim=True).squeeze().cpu().numpy()
    err_t2 = torch.mean((t2 - recon_t2)**2, dim=1, keepdim=True).squeeze().cpu().numpy()
    change_map = np.abs(err_t1 - err_t2)
    change_pred = (change_map > CHANGE_THRESHOLD).astype(np.uint8)

    t2_img = t2[0, :3].cpu().numpy().transpose(1, 2, 0)
    t2_img = np.clip(t2_img / (t2_img.max() + 1e-10), 0, 1)

    axes[i, 0].imshow(t2_img)
    axes[i, 0].set_title('After (T2)')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(err_t2, cmap='hot')
    axes[i, 1].set_title('Recon Error T2')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(change_map, cmap='hot')
    axes[i, 2].set_title('Change Map')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(mask_gt, cmap='gray')
    axes[i, 3].set_title('Ground Truth')
    axes[i, 3].axis('off')

    axes[i, 4].imshow(change_pred, cmap='gray')
    f1 = compute_f1(torch.from_numpy(change_pred).float().unsqueeze(0), torch.from_numpy(mask_gt).float().unsqueeze(0))
    axes[i, 4].set_title(f'Prediction (F1: {f1:.3f})')
    axes[i, 4].axis('off')

plt.suptitle('Self-Supervised Change Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions_selfsupervised.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Sensitivity

In [ ]:
model.eval()
all_f1_by_thresh = {}

for thresh in np.arange(0.05, 0.50, 0.05):
    f1s = []
    with torch.no_grad():
        for batch in val_loader:
            t1 = batch['t1'].to(device)
            t2 = batch['t2'].to(device)
            masks = batch['mask'].to(device)

            recon_t1 = model(t1)
            recon_t2 = model(t2)

            err_t1 = torch.mean((t1 - recon_t1)**2, dim=1, keepdim=True)
            err_t2 = torch.mean((t2 - recon_t2)**2, dim=1, keepdim=True)
            change_map = torch.abs(err_t1 - err_t2)
            change_pred = (change_map > thresh).float()

            f1s.append(compute_f1(change_pred, masks))

    all_f1_by_thresh[thresh] = np.mean(f1s)

best_thresh = max(all_f1_by_thresh, key=all_f1_by_thresh.get)
print(f"Best threshold: {best_thresh:.2f} (F1: {all_f1_by_thresh[best_thresh]:.4f})")

plt.figure(figsize=(8, 5))
plt.plot(list(all_f1_by_thresh.keys()), list(all_f1_by_thresh.values()), 'o-', linewidth=2)
plt.axvline(best_thresh, color='r', linestyle='--', label=f'Best: {best_thresh:.2f}')
plt.xlabel('Change Threshold')
plt.ylabel('F1 Score')
plt.title('Threshold vs F1 Score (Self-Supervised)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_sensitivity_selfsupervised.png', dpi=150)
plt.show()

## Summary

- **No labels used** — trained purely on reconstruction loss
- Expected performance: F1 ~0.35-0.50 (lower than supervised, but no annotation needed)
- Best model saved to `autoencoder_best.pt`
- Key value: demonstrates what's possible when labeled change data isn't available
- Next: `04_patna_case_study.ipynb` to apply both models to real Patna data